# S44_03 — Transformer Architecture

## The full transformer block

One transformer layer = **multi-head attention** + **position-wise feed-forward network**, each wrapped with a **residual connection** and **layer normalisation**:

```
x → LayerNorm → MultiHeadAttention → (+x) → LayerNorm → FFN → (+x) → output
```

The residual connections allow gradients to flow directly to early layers (same idea as ResNets).
Layer normalisation stabilises activations across the sequence dimension.

In [ ]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True, dropout=dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),          # GELU is preferred over ReLU in modern transformers
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, causal=False):
        S = x.shape[1]
        mask = None
        if causal:
            # Upper-triangular mask → attend only to past tokens
            mask = torch.triu(torch.ones(S, S, device=x.device), diagonal=1).bool()

        # Pre-norm style (used by modern LLMs including Llama)
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x), attn_mask=mask)
        x = x + self.drop(attn_out)           # residual
        x = x + self.drop(self.ff(self.norm2(x)))  # residual
        return x


block = TransformerBlock(d_model=256, num_heads=8, d_ff=1024)
x = torch.randn(2, 10, 256)  # batch=2, seq=10
print(f'Input:  {x.shape}')
print(f'Output: {block(x).shape}')  # shape unchanged — transformer is a sequence-to-sequence map

## Three transformer variants

| Architecture | Components | Pre-training objective | Best for |
|---|---|---|---|
| **Encoder-only** (BERT, RoBERTa) | Stack of bidirectional encoder blocks | Masked Language Modelling (MLM) | Classification, NER, embeddings |
| **Decoder-only** (GPT, Llama, Mistral) | Stack of causal decoder blocks | Causal Language Modelling (CLM) | Text generation, reasoning, chat |
| **Encoder-Decoder** (T5, BART) | Encoder stack + cross-attending decoder | Sequence-to-sequence (span masking) | Translation, summarisation, Q&A |

## The feed-forward network

The FFN in each block is applied **independently to each token position**. It has 2 linear layers with a non-linearity:
- Classic: `Linear → ReLU → Linear`
- Modern (BERT, GPT-2): `Linear → GELU → Linear`  
- Very modern (Llama): `SwiGLU` gated activation

`d_ff` is typically `4 × d_model`. This is where most of the model's parameters live — the FFN stores factual knowledge.

## A minimal GPT-style language model

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)  # learned positional embeddings
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids):
        B, S = token_ids.shape
        positions = torch.arange(S, device=token_ids.device).unsqueeze(0)
        x = self.token_emb(token_ids) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x, causal=True)    # causal mask for autoregressive generation
        return self.lm_head(self.norm(x))  # logits over vocab


model = TinyGPT(vocab_size=10000, d_model=256, num_heads=8, d_ff=1024, num_layers=4, max_seq_len=512)
tokens = torch.randint(0, 10000, (2, 20))
logits = model(tokens)
print(f'Logits shape: {logits.shape}')  # (2, 20, 10000)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')  # ~10M for this tiny model

Next: [S44_04_positional_encoding.ipynb](./S44_04_positional_encoding.ipynb)
